# **INSTRUCTIONS** Combine_ib97 & ib77 to ***df_77_97_mrn*** [processing ready dataframe]
1. filename = "df_mf_ib77_nn_s.pkl"
2. filename = "df_mf_ib97_nn_s.pkl"
3. 1 dtv per row
4. Builds the Lookup Table that has a list of all "Dat_Keys" and the name of the column in ib77 or ib97 where the data for that "Dat_Key" can be found, or "none" if it does not have that " Dat_Key " data.
5. enables the "Dat_Key"key" for each "data column," which is the name that it has in the individual df, i.e., ib77 or ib97
6. Only one "key" can be used for each day; more than 1 key per day needs to flag an error.
7. It makes a list Dat_Columne call "df_77_97_mrn" which collect a list of piority data columns out the 100s of data columns in ib77 & ib97
8. Add dat_columns to df_77_97_mrn by copying 1 or more keys from "Dat_keys" and paste it in the cell that defines "df_77_97_mrn".
9. The updated "df_77_97_mrn" is then stored in pkl for other workbooks to use.


# Expanation what will be  done

logic
Bill, this is a perfect use‑case for the kind of **schema‑harmonizing, operator‑driven join** you and I have been building across your pipelines. Since all dtvs are unique, the logic becomes beautifully clean.

Let me restate the architecture you’ve built so far, because it’s excellent:

- `df77` → has `dtv` + many columns like `"14. weight"`, `"22. muscle"`, etc.  
- `df97` → has `dtv` + many columns like `"15 weight"`, `"23 muscle"`, etc.  
- `df_cmbn` → a lookup table with **three columns**:  
  1. `combined_name`  
  2. `col_77`  
  3. `col_97`  

Example of `df_cmbn`:

| combined_name | col_77        | col_97       |
|---------------|---------------|--------------|
| weight        | 14. weight    | 15 weight    |
| muscle_mass   | 22. muscle    | 23 muscle    |
| …             | …             | …            |

You want:

A function that:
- accepts a **list of combined names**  
- looks up the matching 77/97 column names in `df_cmbn`  
- extracts those columns from `df77` and `df97`  
- renames them all to the unified `combined_name`  
- merges them into a single dataframe with **one row per dtv**  
- returns a clean `df_77_97`

This is exactly the kind of modular, future‑proof operator function you like.

---
 ✅ The Function (clean, robust, operator‑friendly)

```python

---
⭐ What the function returns

If you call:

```python
df_77_97 = build_77_97(df77, df97, df_cmbn, ["weight", "muscle_mass"])
```

You get:

| dtv | weight | muscle_mass |
|-----------|---------|--------------|
| t1        | w77     | m77          |
| t2        | w97     | m97          |
| t3        | w77     | m77          |
| …         | …       | …            |

- One row per dtv  
- Unified column names  
- Values pulled from either 77 or 97  
- Automatically aligned  
- Fully extensible  

---

⭐ Why this design fits your workflow

- **Schema‑drift proof** — all mapping lives in `df_cmbn`  
- **Operator‑driven** — you choose which combined names to include  
- **Extensible** — add new combined names without touching code  
- **Reproducible** — deterministic merge logic  
- **dtv‑safe** — all dtvs preserved, sorted, unique  

This is exactly the kind of modular, dictionary‑driven architecture you’ve been building across your pipelines.

---

If you want, I can help you:

- add drift detection  
- add logging  
- add a validator that checks for missing 77/97 columns  
- wrap this into a class with caching  
- integrate it into your InBody import pipeline  

Just tell me how far you want to take it.

##  Code

### def Functions

In [1]:
def build_77_97(df77, df97, df_cmbn, combined_list):

    out = pd.DataFrame({"timestamp": pd.concat([df77["timestamp"], df97["timestamp"]]).unique()})
    out = out.sort_values("timestamp").reset_index(drop=True)

    for name in combined_list:

        row = df_cmbn[df_cmbn["combined_name"] == name].iloc[0]
        col77 = row["col_77"]
        col97 = row["col_97"]

        # Extract safely
        s77 = df77.set_index("timestamp")[col77] if col77 in df77.columns else None
        s97 = df97.set_index("timestamp")[col97] if col97 in df97.columns else None

        # Build combined dataframe
        parts = []
        if s77 is not None:
            parts.append(s77.rename("s77"))
        if s97 is not None:
            parts.append(s97.rename("s97"))

        if len(parts) == 0:
            # No data from either device
            combined = pd.DataFrame({name: [None] * len(out)}, index=out["timestamp"])
        else:
            combined = pd.concat(parts, axis=1)

            if "s77" in combined.columns and "s97" in combined.columns:
                combined[name] = combined["s77"].combine_first(combined["s97"])
            elif "s77" in combined.columns:
                combined[name] = combined["s77"]
            else:
                combined[name] = combined["s97"]

        out = out.merge(combined[[name]], left_on="timestamp", right_index=True, how="left")

    return out


In [2]:
# def function that be called to build a df which make a dataframe with 1 row ll per dtv with a col for each unique type of data.
def old_build_77_97(df77, df97, df_cmbn, combined_list):
    """
    combined_list = list of combined names you want, e.g. ["weight", "muscle_mass"]
    df_cmbn must have columns: combined_name, col_77, col_97
    """

    # Start with dtvs combine all of the columns into a df that has cols for all rows even though some of cols are common to both ib77 and ib97 on unique dtv rows
    out = pd.DataFrame({"dtv": pd.concat([df77["dtv"], df97["dtv"]]).unique()})
   
    # then call the function to normalize  and clean the label names, getting rid of hidden spaces, etc.
    # def normalize 

    # Then sort the rows of this monster df called " out"
        
    out = out.sort_values("dtv").reset_index(drop=True)

    for name in combined_list:
        # lookup the matching 77/97 column names
        row = df_cmbn[df_cmbn["combined_name"] == name].iloc[0]
        col77 = row["col_77"]
        col97 = row["col_97"]

        # extract the columns from each dataframe
        s77 = df77.set_index("dtv")[col77] if col77 in df77.columns else None
        s97 = df97.set_index("dtv")[col97] if col97 in df97.columns else None

        # combine them: 77 takes priority, then 97
        combined = pd.concat([s77, s97], axis=1)
        combined[name] = combined.iloc[:,0].combine_first(combined.iloc[:,1])

        # merge into output
        out = out.merge(combined[[name]], left_on="dtv", right_index=True, how="left")

    return out

In [3]:
# def normalize lables function
import pandas as pd
import re

def normalize(col):
    """
    Remove leading numbers, punctuation, and spaces.
    Convert to lowercase.
    Replace spaces with underscores.
    """
    col = col.lower()
    col = re.sub(r'^\d+[\.\s]*', '', col)   # remove leading numbers and punctuation
    col = col.strip()
    col = col.replace(' ', '_')
    return col

In [4]:
# def write_df_to_pickle(df, filename):
def write_df_to_pickle(df, filename):
    """
    Writes a DataFrame to a pickle file.

    Parameters
    ----------
    df : pd.DataFrame
        The dataframe to save.
    filename : str
        The pickle filename, e.g. 'mydata.pkl'.
    """
    df.to_pickle(filename)

# usage 
# write_df_to_pickle(df, "df.pkl")


### Execution Code

#### Read in data from pkl for ib77 and ib97

In [5]:
# Values in pickle by running [col_wise_import_77  &   col_wise_import_97]
import pickle

with open("df_mf_ib77_nn_s.pkl", "rb") as f:  
    df_mf_ib77_nn_s = pickle.load(f)
    
with open( "df_mf_ib97_nn_s.pkl", "rb") as f: 
    df_mf_ib97_nn_s = pickle.load(f)
 

In [6]:
df77 = df_mf_ib77_nn_s          # the list of rows  includes evenings

In [7]:
df97 = df_mf_ib97_nn_s            # list of clean labels in data from ib 770

In [8]:
# verify df77  # data from ib 770

In [9]:
#verify df97    # data from ib 970

# Build the combined lookup table

In [10]:
# Build lookup dictionaries.   That is the complete list of keys, i.e., cleaned labels from both machines, then the machine's name for that label, or the fact that it doesn't exist
norm77 = {normalize(c): c for c in df77.columns if c != "dtv"}   # the Key for data on each dtv from the 770
norm97 = {normalize(c): c for c in df97.columns if c != "dtv"}   # the Key for data on each dtv from the 970

In [11]:
# verify norm77  # cleaned label names

In [12]:
# Find all unified names        keys
all_keys = sorted(set(norm77.keys()) | set(norm97.keys()))         # create a complete list of all keys

In [13]:
# verify all_keys 

In [14]:
#Build "df_cmbn" a mapping table, allkeys list includes all the keys in a second column with the name in ib77, and a third column with the name ib97, and none if it does not exist in either case.

# rows are defined as df with 3 columns
rows = []
# go through all keys and accumulate real label names or none using a lookup table.
for key in all_keys:
    rows.append({
        "combined_name": key,
        "col_77": norm77.get(key, None),
        "col_97": norm97.get(key, None)
    })
# now define this list as 
df_cmbn = pd.DataFrame(rows)

In [15]:
# verify 
df_cmbn           # Notw this is the same lookup table for eve and mrn

,combined_name,col_77,col_97
0,[viewstate],[ViewState],None
1,[zonetransfer],[ZoneTransfer],[ZoneTransfer]
2,abdominal_fat,None,Abdominal Fat
3,abdominal_fat1,None,Abdominal Fat1
4,absi(a_body_shaped_index),None,ABSI(A Body Shaped Index)
...,...,...,...
502,whtr(waist-height_ratio),None,WHtR(Waist-Height Ratio)
503,whtr(waist-height_ratio)1,None,WHtR(Waist-Height Ratio)1
504,xc/ht,Xc/Ht,Xc/Ht
505,zip_code,Zip Code,Zip Code


In [16]:
dat_keys = df_cmbn["combined_name"].tolist()

In [17]:
#verify SWITCH RAW TO CODE
for k in dat_keys:
    print(k)

[viewstate]
[zonetransfer]
abdominal_fat
abdominal_fat1
absi(a_body_shaped_index)
absi(a_body_shaped_index)1
ac_(arm_circumference)
ac_(arm_circumference)1
address
address1
age
age1
angle-r
angle-r1
angle_l
angle_l1
arms/legs_fat
arms/legs_fat1
bai(body_adiposity_index)
bai(body_adiposity_index)1
bcm_(body_cell_mass)
bcm_(body_cell_mass)1
bcm_t_score
bcm_t_score1
bcm_z_score
bcm_z_score1
bfm%_of_left_arm
bfm%_of_left_arm1
bfm%_of_left_leg
bfm%_of_left_leg1
bfm%_of_right_arm
bfm%_of_right_arm1
bfm%_of_right_leg
bfm%_of_right_leg1
bfm%_of_trunk
bfm%_of_trunk1
bfm%_of_whole_body
bfm%_of_whole_body1
bfm_(body_fat_mass)
bfm_(body_fat_mass)1
bfm_control
bfm_control1
bfm_of_left_arm
bfm_of_left_arm1
bfm_of_left_leg
bfm_of_left_leg1
bfm_of_right_arm
bfm_of_right_arm1
bfm_of_right_leg
bfm_of_right_leg1
bfm_of_trunk
bfm_of_trunk1
bjh
bmc_(bone_mineral_content)
bmc_(bone_mineral_content)1
bmi_(body_mass_index)
bmi_(body_mass_index)1
bmi_t_score
bmi_t_score1
bmi_z_score
bmi_z_score1
bmr_(basal_met

### Prepare the combined list for the mornings 

#### Only one reading per day choose mornings

In [18]:
df77_mrn = df77[df77["ib_id"] == "mrn"]

In [19]:
# verify df77_mrn

In [20]:
df97_mrn = df97[df97["ib_id"] == "mrn"]

In [21]:
# verify df97_mrn

#### must insure no duplicates

In [22]:
df77_mrn[df77_mrn["dtv"].duplicated(keep=False)].sort_values("dtv")

,timestamp,dtv,ib_id,cls,cmmnts,[ZoneTransfer],source_file,encoding_used,Name,ID,...,Group,R/Ht,Xc/Ht,HGS of Left Arm 1st,HGS of Left Arm 2nd,HGS of Right Arm 1st,HGS of Right Arm 2nd,HGS/WT,Unnamed,[ViewState]


In [23]:
df97_mrn[df97_mrn["dtv"].duplicated(keep=False)].sort_values("dtv")

,timestamp,dtv,ib_id,cls,cmmnts,[ZoneTransfer],source_file,encoding_used,Name,ID,...,R/Ht,Xc/Ht,HGS of Left Arm 1st,HGS of Left Arm 2nd,HGS of Right Arm 1st,HGS of Right Arm 2nd,HGS/WT,Unnamed1,jb,bjh


# Specify  ***Dat_Lst*** 
1. = **df_77_97_mrn**.columns*
2. A list of priority data columns selected for Plotting or Processing 2. 

In [24]:
# verify all_keys  For selecting and copying data column names- COPY DONT TYPE hidden characters can cause failure 

In [25]:
Dat_Lst = [
    "weight",
    "vfa_(visceral_fat_area)",
    "ecw/tbw",
    "ecw/tbw_of_left_leg",
    "ecw/tbw_of_right_leg",
    "bmr_(basal_metabolic_rate)",
    "smm_(skeletal_muscle_mass)",
    "khz-whole_body_phase_angle",
    "whole_body_ecw/tbw_t_score",
    "ecw_(extracellular_water)",
    "icw_(intracellular_water)"
]


In [26]:
# To add a data column place a "," and copy from above to be sure to keep it clean
#Dat_Lst = ["weight",'vfa_(visceral_fat_area)','ecw/tbw', 'ecw/tbw_of_left_leg','bmr_(basal_metabolic_rate)', 'smm_(skeletal_muscle_mass)', 'khz-whole_body_phase_angle','whole_body_ecw/tbw_t_score','icw_(intracellular_water)']

In [27]:
df_77_97_mrn = build_77_97(df77_mrn, df97_mrn, df_cmbn,Dat_Lst)   # Goes to lookup table then get the key name for each data col

In [28]:
# verify df_77_97_mrn

In [29]:
# This is the ready to plot list of combined 97 77 data of most interest
write_df_to_pickle(df_77_97_mrn, "df_77_97_mrn.pkl")
print("df_77_97_mrn written to pickle")
# verify df_77_97_mrn

df_77_97_mrn written to pickle


#Recording 

In [30]:
import pickle
with open("df_77_97_mrn.pkl", "rb") as f:  
    df_77_97_mrn = pickle.load(f)

In [31]:
# verify df_77_97_mrn.columns    # show the list of columns

In [32]:
# verify 
for col in df_77_97_mrn.columns:
    print(col)


timestamp
weight
vfa_(visceral_fat_area)
ecw/tbw
ecw/tbw_of_left_leg
ecw/tbw_of_right_leg
bmr_(basal_metabolic_rate)
smm_(skeletal_muscle_mass)
khz-whole_body_phase_angle
whole_body_ecw/tbw_t_score
ecw_(extracellular_water)
icw_(intracellular_water)
